# Herogod — open AI image studio

Chat-based image editing with **InstructPix2Pix**, **SDXL**, optional **InsightFace face swap**, and optional free-tier API providers.

**Before starting:** choose *Runtime → Change runtime type → T4 GPU*. Free Colab GPU availability and runtime duration are not guaranteed. Local open models need no AI API key. The access token generated below only protects your temporary Herogod tunnel.

Use images you have rights to edit. Face swaps require permission from recognizable people; never create non-consensual intimate imagery, sexual imagery of minors, or deceptive content. Herogod does not bypass hosted-provider safeguards.

In [ ]:
# Locate the source files, or clone them when this notebook was opened directly from GitHub.
from pathlib import Path
import os
import subprocess

if (Path.cwd() / 'server.py').is_file():
    PROJECT_DIR = Path.cwd()
else:
    PROJECT_DIR = Path('/content/Herogod')
    if not (PROJECT_DIR / 'server.py').is_file():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/tao-mihiranga/Herogod.git', str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)
try:
    import torch
    print('GPU available:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch will be installed in the next setup cell.')

In [ ]:
# @title Runtime choices { display-mode: "form" }
MODEL_ID = 'timbrooks/instruct-pix2pix' # @param ['timbrooks/instruct-pix2pix', 'stabilityai/stable-diffusion-xl-base-1.0']
MAX_IMAGE_SIZE = 768 # @param {type:'integer'}
ENABLE_FACE_SWAP = False # @param {type:'boolean'}

import os
import secrets

os.environ['HEROGOD_MODEL'] = MODEL_ID
os.environ['HEROGOD_MAX_IMAGE_SIZE'] = str(MAX_IMAGE_SIZE)
os.environ['HEROGOD_CPU_OFFLOAD'] = '1' if 'stable-diffusion-xl' in MODEL_ID else '0'
os.environ['HEROGOD_API_TOKEN'] = secrets.token_urlsafe(30)

# Optional: add HF_TOKEN and/or GEMINI_API_KEY in Colab Secrets and enable notebook access.
# Values are never printed. Leave them absent to use only local open models.
try:
    from google.colab import userdata
    for secret_name in ('HF_TOKEN', 'GEMINI_API_KEY'):
        try:
            secret_value = userdata.get(secret_name)
            if secret_value: os.environ[secret_name] = secret_value
        except Exception:
            pass
except ImportError:
    pass

print('Configuration ready. Provider keys loaded:', ', '.join(k for k in ('HF_TOKEN', 'GEMINI_API_KEY') if os.getenv(k)) or 'none — local only')

In [ ]:
# Install the web server, API adapters, and local Diffusers dependencies.
# Colab's existing PyTorch build is retained when it meets requirements-local.txt.
!pip -q install -r requirements.txt -r requirements-local.txt

if ENABLE_FACE_SWAP:
    !pip -q install -r requirements-face.txt
    !python scripts/setup_face_swap.py

print('Installation complete.')

In [ ]:
# Start Herogod on port 8000. Model weights load lazily on the first edit.
import socket
import subprocess
import sys
import time

server_log = open('/tmp/herogod-server.log', 'w')
server_process = subprocess.Popen(
    [sys.executable, 'server.py', '--host', '0.0.0.0', '--port', '8000'],
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
for _ in range(40):
    if server_process.poll() is not None:
        raise RuntimeError(Path('/tmp/herogod-server.log').read_text())
    try:
        with socket.create_connection(('127.0.0.1', 8000), timeout=1):
            break
    except OSError:
        time.sleep(0.5)
else:
    raise RuntimeError('Server did not open port 8000. Check /tmp/herogod-server.log.')
print('Herogod backend is running.')

In [ ]:
# Create a temporary HTTPS tunnel. No Cloudflare account or token is required.
import platform
import re
import stat
import subprocess
import time
import urllib.request

if platform.machine() not in ('x86_64', 'AMD64'):
    raise RuntimeError('This launcher expects a 64-bit Intel/AMD Colab runtime.')
cloudflared = Path('/tmp/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)
tunnel_log = open('/tmp/herogod-tunnel.log', 'w')
tunnel_process = subprocess.Popen(
    [str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)
public_url = None
for _ in range(90):
    if tunnel_process.poll() is not None: break
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', Path('/tmp/herogod-tunnel.log').read_text())
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)
if not public_url:
    raise RuntimeError('Tunnel did not start. Log:\n' + Path('/tmp/herogod-tunnel.log').read_text())

print('\n' + '─' * 66)
print('HEROGOD IS READY')
print('Open the studio:', public_url)
print('Backend URL:   ', public_url)
print('Access token:  ', os.environ['HEROGOD_API_TOKEN'])
print('─' * 66)
print('Open the studio URL, then paste the access token in Connection settings.')
print('To use a separately opened index.html, paste both the Backend URL and token.')
print('Keep this notebook running. The tunnel stops when the runtime stops.')

In [ ]:
# Optional diagnostics: shows provider readiness without exposing any keys.
import json
import urllib.request
request = urllib.request.Request(public_url + '/api/health', headers={'X-Herogod-Token': os.environ['HEROGOD_API_TOKEN']})
with urllib.request.urlopen(request, timeout=30) as response:
    print(json.dumps(json.load(response), indent=2))

## Practical notes

- **First edit:** model download and initialization can take several minutes. Watch `/tmp/herogod-server.log` if needed.
- **InstructPix2Pix:** best default for a free T4; describe direct changes such as “make the background a rainy neon street.”
- **SDXL img2img:** better for broader restyling; use *Change strength* to control how far it moves from the source. A larger image or high step count can exhaust GPU memory.
- **Face swap:** enable it in the configuration cell before installation. The target supplies the scene; the reference supplies the identity. The largest detected face is used. Review InsightFace's non-commercial research licensing.
- **Hosted APIs:** add `HF_TOKEN` or `GEMINI_API_KEY` through Colab Secrets before running the configuration cell. Provider quotas, licenses, availability, and safety policies still apply.
- **Privacy:** images are processed in memory and returned to your browser; Herogod adds no chat/image database. Large multipart uploads can use short-lived framework temporary files, which are closed after reading. Your tunnel provider and any hosted AI provider have their own data policies.

To stop the backend and tunnel, run the final cell or stop/reset the Colab runtime.

In [ ]:
# Optional: stop processes without resetting the whole runtime.
for process_name in ('tunnel_process', 'server_process'):
    process = globals().get(process_name)
    if process and process.poll() is None:
        process.terminate()
        print('Stopped', process_name)